In [1]:
# ============================================================
# M5.3 — COVER-DOMINATION CLOSURE ROUTE
# ============================================================
#
# Purpose:
#   Reduce the remaining M5 ceiling to two clean analytic lemmas.
#
# Lemma A candidate:
#   For every L >= 4 and t > 0,
#
#       Phi_L(t)^4 - L^-4 <= q(t)^4,
#
#   where
#       Phi_L(t) = e^{-2t} sum_{j in Z} I_{jL}(2t)
#       q(t)     = e^{-2t} I_0(2t).
#
#   This is a finite-torus heat-kernel return probability minus stationarity
#   bounded by the infinite-cover return probability.
#
# Lemma B candidate:
#   The infinite-lattice renormalized constant
#
#       C_inf(L)
#       =
#       Y_inf(mu_L^2)
#       - A log(L/4)
#       - (A/2) log beta(L)
#
#   is < Ccrit for all L >= 4.
#
# If Lemma A + Lemma B are proved, then:
#
#       C_L <= C_inf(L) < Ccrit
#       T_C < 1/8
#       N*_C >= 8
#       T_full < 9/64
#       N* >= 7
#
# This block numerically audits both lemmas and computes the exact margin.
# ============================================================

import math
import time
import numpy as np

try:
    from scipy.special import ive, exp1
    from scipy.integrate import quad
except Exception as e:
    raise RuntimeError("SciPy is required. In Colab, run: !pip install scipy") from e

print("=" * 100)
print("M5.3 COVER-DOMINATION CLOSURE ROUTE")
print("=" * 100)

# ------------------------------------------------------------
# Constants
# ------------------------------------------------------------

BETA0 = 5.6
GAMMA = 11.0 / (8.0 * math.pi**2)
A = 3.0 / (32.0 * math.pi**2)

# From M5.2
C_CRIT = 0.036241888089666857

print(f"BETA0   = {BETA0:.18f}")
print(f"GAMMA   = {GAMMA:.18f}")
print(f"A       = {A:.18f}")
print(f"C_CRIT  = {C_CRIT:.18f}")
print()

# ------------------------------------------------------------
# AF diagonal
# ------------------------------------------------------------

def beta_x(x):
    return BETA0 + GAMMA * x

def beta_L(L):
    return BETA0 + GAMMA * math.log(L / 4.0)

def mu2_x(x):
    # L = 4 exp(x)
    L = 4.0 * math.exp(x)
    return 48.0 / (beta_x(x) * L * L)

def mu2_L(L):
    return 48.0 / (beta_L(L) * L * L)

# ------------------------------------------------------------
# Heat kernels
# ------------------------------------------------------------

def q_infinite(t):
    """
    Infinite 1D lattice heat-kernel return probability:
        q(t)=e^{-2t} I_0(2t).
    scipy ive(0,2t)=e^{-2t}I_0(2t).
    """
    return float(ive(0, 2.0 * t))

def Phi_L(L, t, rel_cut=1e-16):
    """
    Exact 1D cycle heat-kernel return probability using image sum:
        Phi_L(t) = sum_{j in Z} e^{-2t} I_{jL}(2t).
    """
    out = float(ive(0, 2.0 * t))
    j = 1

    while True:
        term = 2.0 * float(ive(j * L, 2.0 * t))
        out_new = out + term

        if term <= rel_cut * max(abs(out_new), 1e-300):
            return out_new

        out = out_new
        j += 1

        if j > 300000:
            raise RuntimeError(f"Phi_L image sum runaway: L={L}, t={t}, out={out}")

# ------------------------------------------------------------
# Candidate Lemma A pointwise audit
# ------------------------------------------------------------

def audit_cover_domination_for_L(L):
    """
    Numerically audits:
        Phi_L(t)^4 - L^-4 <= q(t)^4
    over a multiscale t grid.
    """
    # t windows:
    #   small/UV absolute scale
    #   torus mixing scale t ~ L^2
    #   post-mixing scale
    abs_grid = np.logspace(-8, 4, 260)
    scaled_grid = (L * L) * np.logspace(-8, 2.5, 360)
    t_grid = np.unique(np.concatenate([abs_grid, scaled_grid]))

    worst_gap = -1e300
    worst_ratio = -1e300
    worst_t = None
    bad = []

    for t in t_grid:
        phi = Phi_L(L, float(t))
        q = q_infinite(float(t))

        lhs = phi**4 - L**-4
        rhs = q**4
        gap = lhs - rhs

        # ratio is diagnostic only
        ratio = lhs / rhs if rhs > 0 else float("nan")

        if gap > worst_gap:
            worst_gap = gap
            worst_ratio = ratio
            worst_t = float(t)

        if gap > 1e-11 * max(1.0, abs(rhs), abs(lhs)):
            bad.append((float(t), lhs, rhs, gap, ratio))
            break

    return {
        "L": L,
        "pass": len(bad) == 0,
        "worst_gap": worst_gap,
        "worst_ratio": worst_ratio,
        "worst_t": worst_t,
        "bad": bad[:1],
    }

print("LEMMA A NUMERICAL AUDIT: Phi_L^4 - L^-4 <= q^4")
print("-" * 100)

cover_Ls = [4, 5, 6, 8, 12, 16, 24, 32, 47, 64, 96, 128, 192, 256, 384, 512]
cover_results = []

t0 = time.time()
for L in cover_Ls:
    res = audit_cover_domination_for_L(L)
    cover_results.append(res)

    status = "PASS" if res["pass"] else "FAIL"
    print(
        f"L={L:4d}  {status:4s}  "
        f"worst_gap={res['worst_gap']:.3e}  "
        f"worst_ratio={res['worst_ratio']:.12f}  "
        f"worst_t={res['worst_t']:.6e}"
    )

print(f"cover audit elapsed = {time.time() - t0:.2f}s")
print(f"all tested cover-domination pass? {all(r['pass'] for r in cover_results)}")
print()

# ------------------------------------------------------------
# Infinite-lattice Y_inf with asymptotic subtraction
# ------------------------------------------------------------

def Y_inf(mu2, T0=64.0):
    """
    Computes:
        Y_inf(mu2) = (3/4) ∫_0^∞ t e^{-mu2 t} q(t)^4 dt.

    Uses asymptotic subtraction for the logarithmic tail:
        q(t)^4 ~ 1/(16*pi^2*t^2)
        t*q(t)^4 ~ 1/(16*pi^2*t)

    Therefore:
        ∫_{T0}^∞ e^{-mu2 t} (1/t) dt = E1(mu2*T0).

    The residual is integrable at infinity.
    """
    c = 1.0 / (16.0 * math.pi**2)

    def f0(t):
        q = q_infinite(t)
        return t * math.exp(-mu2 * t) * q**4

    v0, e0 = quad(f0, 0.0, T0, epsabs=2e-13, epsrel=2e-12, limit=400)

    def fres(t):
        q = q_infinite(t)
        return math.exp(-mu2 * t) * (t * q**4 - c / t)

    vres, eres = quad(fres, T0, math.inf, epsabs=2e-13, epsrel=2e-11, limit=500)

    vlog = c * float(exp1(mu2 * T0))

    Y = 0.75 * (v0 + vres + vlog)
    err = 0.75 * (abs(e0) + abs(eres))
    return Y, err

def C_inf_from_x(x):
    b = beta_x(x)
    mu2 = mu2_x(x)
    Y, err = Y_inf(mu2)
    C = Y - A * x - 0.5 * A * math.log(b)
    return {
        "x": x,
        "L_proxy": 4.0 * math.exp(x),
        "beta": b,
        "mu2": mu2,
        "Y_inf": Y,
        "C_inf": C,
        "err": err,
        "margin": C_CRIT - C,
    }

print("=" * 100)
print("LEMMA B NUMERICAL AUDIT: infinite-lattice renormalized constant")
print("-" * 100)

# Important x values:
#   0                  L=4
#   log(47/4)          finite corrected peak
#   log(4096/4)        finite scan endpoint
#   30.774...          corrected ceiling maximum location
#   large values       asymptotic constant
x_values = sorted(set([
    0.0,
    math.log(5/4),
    math.log(6/4),
    math.log(8/4),
    math.log(12/4),
    math.log(16/4),
    math.log(32/4),
    math.log(47/4),
    math.log(64/4),
    math.log(128/4),
    math.log(256/4),
    math.log(512/4),
    math.log(1024/4),
    math.log(2048/4),
    math.log(4096/4),
    10.0,
    20.0,
    30.774163829,
    36.271356,
    50.0,
    75.0,
    100.0,
]))

inf_results = []
t1 = time.time()

print(f"{'x':>12} {'L_proxy':>15} {'beta':>12} {'C_inf':>18} {'margin':>18} {'quad_err':>12}")
for x in x_values:
    r = C_inf_from_x(x)
    inf_results.append(r)
    print(
        f"{r['x']:12.6f} "
        f"{r['L_proxy']:15.4e} "
        f"{r['beta']:12.6f} "
        f"{r['C_inf']:18.12f} "
        f"{r['margin']:18.12f} "
        f"{r['err']:12.3e}"
    )

print(f"infinite audit elapsed = {time.time() - t1:.2f}s")
print()

max_inf = max(inf_results, key=lambda r: r["C_inf"])
print("INFINITE-LATTICE SUMMARY")
print("-" * 100)
print(f"max audited C_inf      = {max_inf['C_inf']:.18f}")
print(f"at x                   = {max_inf['x']:.9f}")
print(f"L proxy                = {max_inf['L_proxy']:.6e}")
print(f"C_CRIT                 = {C_CRIT:.18f}")
print(f"margin                 = {C_CRIT - max_inf['C_inf']:.18f}")
print()

# ------------------------------------------------------------
# Optional dense comparison to finite C_L if M5.2 data exist
# ------------------------------------------------------------

try:
    corrected
    has_corrected = True
except NameError:
    has_corrected = False

if has_corrected:
    print("=" * 100)
    print("FINITE C_L VS INFINITE COVER BOUND AT MATCHED L")
    print("-" * 100)
    print(f"{'L':>6} {'C_L finite':>16} {'C_inf':>16} {'gap Cinf-CL':>16}")

    for L in [4, 5, 6, 8, 12, 16, 32, 47, 64, 128, 256, 512, 1024, 2048, 4096]:
        z = next((r for r in corrected if r["L"] == L), None)
        if z is None:
            continue

        x = math.log(L / 4.0)
        ri = C_inf_from_x(x)
        print(
            f"{L:6d} "
            f"{z['C_L']:16.12f} "
            f"{ri['C_inf']:16.12f} "
            f"{(ri['C_inf'] - z['C_L']):16.12f}"
        )

    print()

# ------------------------------------------------------------
# Ceiling consequence using an infinite-lattice C envelope
# ------------------------------------------------------------

def TC_envelope_from_C(x, C):
    b = beta_x(x)
    return 36.0 * (A * x + 0.5 * A * math.log(b) + C) / (b * b)

def maximize_envelope(C, xmax=200.0):
    lo, hi = 0.0, xmax

    for _ in range(240):
        m1 = lo + (hi - lo) / 3.0
        m2 = hi - (hi - lo) / 3.0

        if TC_envelope_from_C(m1, C) < TC_envelope_from_C(m2, C):
            lo = m1
        else:
            hi = m2

    xs = 0.5 * (lo + hi)
    return xs, beta_x(xs), TC_envelope_from_C(xs, C)

print("=" * 100)
print("CLOSURE CONSEQUENCE IF COVER DOMINATION IS PROVED")
print("-" * 100)

for Csafe in [
    max_inf["C_inf"],
    0.013,
    0.015,
    0.020,
    0.025,
    0.030,
    C_CRIT,
]:
    xs, bs, tb = maximize_envelope(Csafe)
    print(
        f"Csafe={Csafe:.12f}  "
        f"x*={xs:.6f}  "
        f"beta*={bs:.6f}  "
        f"Tbar={tb:.12f}  "
        f"floor(1/Tbar)={int(1.0 // tb):3d}"
    )

print()
print("M5.3 STATUS")
print("-" * 100)
print("The remaining proof can now be stated as two focused lemmas:")
print()
print("Lemma A, cover domination:")
print("    Phi_L(t)^4 - L^-4 <= q(t)^4 for all L >= 4 and t > 0.")
print()
print("Lemma B, infinite-lattice renormalized bound:")
print("    Y_inf(mu_L^2) - A log(L/4) - (A/2) log beta(L) <= 0.013")
print("    is already far stronger than needed; Ccrit is about 0.03624.")
print()
print("If Lemma A and Lemma B are proved, then:")
print("    C_L <= 0.013 < Ccrit")
print("    T_C < 1/8")
print("    N*_C >= 8")
print("    T_full < 9/64")
print("    N* >= 7")
print("=" * 100)

M5.3 COVER-DOMINATION CLOSURE ROUTE
BETA0   = 5.599999999999999645
GAMMA   = 0.139316627508214441
A       = 0.009498860966469166
C_CRIT  = 0.036241888089666857

LEMMA A NUMERICAL AUDIT: Phi_L^4 - L^-4 <= q^4
----------------------------------------------------------------------------------------------------
L=   4  PASS  worst_gap=-6.333e-11  worst_ratio=-0.000000136965  worst_t=1.000000e+04
L=   5  PASS  worst_gap=-6.333e-11  worst_ratio=-0.000000010272  worst_t=1.000000e+04
L=   6  PASS  worst_gap=-4.886e-11  worst_ratio=-0.000000024407  worst_t=1.138420e+04
L=   8  PASS  worst_gap=-1.546e-11  worst_ratio=0.000000112203  worst_t=2.023858e+04
L=  12  PASS  worst_gap=-3.054e-12  worst_ratio=-0.000000024408  worst_t=4.553680e+04
L=  16  PASS  worst_gap=-9.663e-13  worst_ratio=0.000000196357  worst_t=8.095431e+04
L=  24  PASS  worst_gap=-1.909e-13  worst_ratio=0.000000059910  worst_t=1.821472e+05
L=  32  PASS  worst_gap=-6.039e-14  worst_ratio=0.000000182332  worst_t=3.238172e+05
L=  47 

/tmp/ipykernel_5489/2137306929.py:229: IntegrationWarning: The occurrence of roundoff error is detected, which prevents 
  the requested tolerance from being achieved.  The error may be 
  underestimated.
  vres, eres = quad(fres, T0, math.inf, epsabs=2e-13, epsrel=2e-11, limit=500)
